# Generate Large-Scale Simulation Input
Creates object_input_sim.csv with many random objects

In [43]:
import numpy as np
import pandas as pd
from datetime import datetime

## Configuration

In [44]:
# Number of objects to generate
NUM_OBJECTS = 10000

# Space parameters
MAX_DISTANCE = 5e12  # Maximum distance from Sun (meters)
MIN_DISTANCE = 1e10  # Minimum distance from Sun (meters)

# Mass range (kg)
MIN_MASS = 1e20  # Small asteroid
MAX_MASS = 1e27  # Large planet

# Gravitational constant
G = 6.67430e-11
SUN_MASS = 1.989e30

print(f"Generating {NUM_OBJECTS:,} objects...")
print(f"Distance range: {MIN_DISTANCE:.2e} to {MAX_DISTANCE:.2e} meters")
print(f"Mass range: {MIN_MASS:.2e} to {MAX_MASS:.2e} kg")

Generating 10,000 objects...
Distance range: 1.00e+10 to 5.00e+12 meters
Mass range: 1.00e+20 to 1.00e+27 kg


## Generate Objects

In [45]:
print("\nGenerating random objects...")
start_time = datetime.now()

# Random masses
print("  [1/5] Generating masses...")
masses = np.random.uniform(MIN_MASS, MAX_MASS, NUM_OBJECTS)
print(f"    ✓ Done ({(datetime.now() - start_time).total_seconds():.2f}s)")

# Random positions in circular disk (2D for simplicity)
print("  [2/5] Generating positions...")
distances = np.random.uniform(MIN_DISTANCE, MAX_DISTANCE, NUM_OBJECTS)
angles = np.random.uniform(0, 2*np.pi, NUM_OBJECTS)
pos_x = distances * np.cos(angles)
pos_y = distances * np.sin(angles)
pos_z = np.zeros(NUM_OBJECTS)  # Flat disk
print(f"    ✓ Done ({(datetime.now() - start_time).total_seconds():.2f}s)")

# Calculate orbital velocities (circular orbit approximation)
print("  [3/5] Calculating orbital velocities...")
# v = sqrt(G * M_sun / r)
orbital_speeds = np.sqrt(G * SUN_MASS / distances)
print(f"    ✓ Done ({(datetime.now() - start_time).total_seconds():.2f}s)")

# Velocity perpendicular to position (circular orbit)
print("  [4/5] Generating velocity vectors...")
vel_x = -orbital_speeds * np.sin(angles)
vel_y = orbital_speeds * np.cos(angles)
vel_z = np.zeros(NUM_OBJECTS)
print(f"    ✓ Done ({(datetime.now() - start_time).total_seconds():.2f}s)")

# Zero initial acceleration
print("  [5/5] Initializing accelerations...")
acc_x = np.zeros(NUM_OBJECTS)
acc_y = np.zeros(NUM_OBJECTS)
acc_z = np.zeros(NUM_OBJECTS)

elapsed = (datetime.now() - start_time).total_seconds()
print(f"\n✓ Generated {NUM_OBJECTS:,} objects in {elapsed:.2f} seconds")


Generating random objects...
  [1/5] Generating masses...
    ✓ Done (0.00s)
  [2/5] Generating positions...
    ✓ Done (0.00s)
  [3/5] Calculating orbital velocities...
    ✓ Done (0.00s)
  [4/5] Generating velocity vectors...
    ✓ Done (0.00s)
  [5/5] Initializing accelerations...

✓ Generated 10,000 objects in 0.00 seconds


## Add Sun at center

In [46]:
# Prepend Sun to arrays
names = ['Sun'] + [f'Object{i}' for i in range(NUM_OBJECTS)]
masses = np.insert(masses, 0, SUN_MASS)
pos_x = np.insert(pos_x, 0, 0)
pos_y = np.insert(pos_y, 0, 0)
pos_z = np.insert(pos_z, 0, 0)
vel_x = np.insert(vel_x, 0, 0)
vel_y = np.insert(vel_y, 0, 0)
vel_z = np.insert(vel_z, 0, 0)
acc_x = np.insert(acc_x, 0, 0)
acc_y = np.insert(acc_y, 0, 0)
acc_z = np.insert(acc_z, 0, 0)

print(f"\nTotal objects: {len(names):,}")


Total objects: 10,001


## Save to CSV

In [47]:
print("\nWriting to CSV...")
start_time = datetime.now()

# Write CSV manually to avoid pandas quoting
output_file = 'object_input_sim.csv'
total = len(names)

print(f"  Writing {total:,} objects to {output_file}...")

with open(output_file, 'w') as f:
    # Write header
    f.write("NAME,MASS,VELOCITY,ACCELERATION,POSITION\n")
    
    # Write each object
    for i in range(total):
        if i % 100000 == 0 and i > 0:
            progress = i / total * 100
            print(f"    Progress: {progress:.1f}% ({i:,}/{total:,})", end='\r')
        
        # Format line
        name = names[i]
        mass = masses[i]
        vel = f"({vel_x[i]},{vel_y[i]},{vel_z[i]})"
        acc = f"({acc_x[i]},{acc_y[i]},{acc_z[i]})"
        pos = f"({pos_x[i]},{pos_y[i]},{pos_z[i]})"
        
        f.write(f"{name},{mass},{vel},{acc},{pos}\n")

elapsed = (datetime.now() - start_time).total_seconds()
print(f"\n  ✓ Done ({elapsed:.2f}s)                    ")

# Check file size
import os
file_size = os.path.getsize(output_file) / (1024**2)
print(f"\n✓ Saved to {output_file}")
print(f"  File size: {file_size:.1f} MB")
print(f"  Total time: {elapsed:.2f} seconds")


Writing to CSV...
  Writing 10,001 objects to object_input_sim.csv...

  ✓ Done (0.03s)                    

✓ Saved to object_input_sim.csv
  File size: 1.3 MB
  Total time: 0.03 seconds


## Preview Data

In [48]:
print("\nPreview (first 5 objects):")
with open('object_input_sim.csv', 'r') as f:
    for i, line in enumerate(f):
        if i < 6:  # Header + 5 objects
            print(f"  {line.strip()}")

print("\nStatistics:")
print(f"  Total objects: {len(names):,}")
print(f"  Average mass: {np.mean(masses):.2e} kg")
print(f"  Average distance: {np.sqrt(pos_x**2 + pos_y**2).mean():.2e} meters")
print(f"  Average velocity: {np.sqrt(vel_x**2 + vel_y**2).mean():.2e} m/s")

print("\n✓ Done! Use object_input_sim.csv for large-scale simulation")


Preview (first 5 objects):
  NAME,MASS,VELOCITY,ACCELERATION,POSITION
  Sun,1.989e+30,(0.0,0.0,0.0),(0.0,0.0,0.0),(0.0,0.0,0.0)
  Object0,5.434844749727643e+26,(3044.0837407657996,5273.73016321522,0.0),(0.0,0.0,0.0),(3100788113986.5522,-1789825870725.1772,0.0)
  Object1,5.8229553149582846e+26,(-8377.408962772677,-1889.8702619259618,0.0),(0.0,0.0,0.0),(-396101583826.3063,1755837437821.4065,0.0)
  Object2,8.317567683029358e+26,(942.9439931835537,-6456.752560287077,0.0),(0.0,0.0,0.0),(-3085066783629.773,-450543081066.1978,0.0)
  Object3,8.398142629888006e+26,(-3350.8178304655817,-5110.317194243575,0.0),(0.0,0.0,0.0),(-2972822235637.139,1949269561035.9185,0.0)

Statistics:
  Total objects: 10,001
  Average mass: 6.99e+26 kg
  Average distance: 2.51e+12 meters
  Average velocity: 9.87e+03 m/s

✓ Done! Use object_input_sim.csv for large-scale simulation
